# 🧠 Brain Tumor Classification using CNN

**Dataset:** Brain Tumor MRI Dataset (Kaggle) — 4 classes: `glioma`, `meningioma`, `notumor`, `pituitary`

**Model:** Custom CNN with TensorFlow/Keras  
**Author:** ponugupati ashwith chowdary

## Step 1 — Install & Import Libraries

In [ ]:
!pip install -q numpy pandas matplotlib scikit-learn tensorflow streamlit pyngrok

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print('TensorFlow version:', tf.__version__)

## Step 2 — Upload & Extract Dataset

> Upload `archive (5).zip` from [Kaggle Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print(os.listdir('/content/'))

## Step 3 — Explore the Dataset

In [ ]:
TRAIN_DIR = '/content/Training'
TEST_DIR  = '/content/Testing'

for folder in ['Training', 'Testing']:
    print(f'\n{folder} Dataset:')
    for category in sorted(os.listdir(f'/content/{folder}')):
        count = len(os.listdir(f'/content/{folder}/{category}'))
        print(f'  {category}: {count} images')

## Step 4 — Load & Preprocess Data

In [ ]:
IMG_SIZE   = (128, 128)
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=42
)
test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_data.class_names
print('Class names:', class_names)

# Normalize pixel values to [0, 1]
norm_layer = tf.keras.layers.Rescaling(1.0 / 255)
train_data = train_data.map(lambda x, y: (norm_layer(x), y)).cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
test_data  = test_data.map(lambda x, y: (norm_layer(x), y)).cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
# Visualize sample images
for images, labels in train_data.take(1):
    plt.figure(figsize=(14, 6))
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(class_names[labels[i].numpy()], fontsize=12)
        plt.axis('off')
    plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Count images per class
label_counts = np.zeros(4, dtype=int)
for _, labels in train_data:
    label_counts += np.bincount(labels.numpy(), minlength=4)

plt.figure(figsize=(8, 4))
plt.bar(class_names, label_counts, color=['#e74c3c','#3498db','#2ecc71','#f39c12'])
plt.title('Training Images per Class')
plt.ylabel('Count')
for i, v in enumerate(label_counts):
    plt.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5 — Build the CNN Model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    # Data augmentation
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),

    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Classifier
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation='softmax')  # ✅ softmax for multi-class
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',  # ✅ for integer labels
    metrics=['accuracy']
)

model.summary()

## Step 6 — Train the Model

In [ ]:
cb_list = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
]

history = model.fit(
    train_data,
    epochs=15,
    validation_data=test_data,
    callbacks=cb_list
)

## Step 7 — Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Validation', marker='s')
axes[0].set_title('Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train', marker='o')
axes[1].plot(history.history['val_loss'], label='Validation', marker='s')
axes[1].set_title('Model Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8 — Evaluate the Model

In [ ]:
test_loss, test_acc = model.evaluate(test_data, verbose=0)
print(f'Test Accuracy : {test_acc * 100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_data])
y_pred = np.argmax(model.predict(test_data, verbose=0), axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title('Confusion Matrix — Brain Tumor Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

## Step 9 — Save & Download the Model

In [ ]:
model.save('/content/brain_tumor_cnn.keras')
print('Model saved:', os.path.exists('/content/brain_tumor_cnn.keras'))

In [ ]:
from google.colab import files
files.download('/content/brain_tumor_cnn.keras')

## Step 10 — Predict on a Single Image

In [ ]:
from google.colab import files
uploaded_img = files.upload()
image_path   = list(uploaded_img.keys())[0]

In [ ]:
img = tf.keras.utils.load_img(image_path, target_size=(128, 128))

img_array = tf.keras.utils.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction     = model.predict(img_array, verbose=0)[0]
predicted_idx  = np.argmax(prediction)
confidence     = prediction[predicted_idx] * 100

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f'Prediction: {class_names[predicted_idx]}\nConfidence: {confidence:.2f}%',
          fontsize=13, fontweight='bold', color='darkgreen')
plt.axis('off')
plt.show()

print('\nAll class probabilities:')
for name, prob in zip(class_names, prediction):
    bar = '█' * int(prob * 30)
    print(f'  {name:12s}: {prob*100:5.1f}%  {bar}')

## Step 11 — Launch Streamlit App via ngrok

In [ ]:
%%writefile /content/app.py
from pathlib import Path
import numpy as np
from PIL import Image
import streamlit as st
import tensorflow as tf

st.set_page_config(page_title='Brain Tumor Classification', page_icon='🧠')

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

@st.cache_resource
def load_model():
    return tf.keras.models.load_model('/content/brain_tumor_cnn.keras')

model = load_model()

st.title('🧠 Brain Tumor Classification')
st.write('Upload a brain MRI image to classify it into one of four categories.')

uploaded_file = st.file_uploader('Choose an MRI image', type=['jpg', 'jpeg', 'png'])
if uploaded_file:
    image = Image.open(uploaded_file).convert('RGB')
    st.image(image, caption='Uploaded MRI Image')

    if st.button('Predict'):
        img_array = np.expand_dims(
            np.array(image.resize((128, 128)), dtype=np.float32) / 255.0, axis=0
        )
        prediction    = model.predict(img_array, verbose=0)[0]
        predicted_idx = int(np.argmax(prediction))
        confidence    = float(prediction[predicted_idx]) * 100

        st.success(f'Predicted Class: **{CLASS_NAMES[predicted_idx]}**')
        st.write(f'Confidence: **{confidence:.2f}%**')

        st.subheader('Class Probabilities')
        for name, prob in zip(CLASS_NAMES, prediction):
            st.progress(float(prob), text=f'{name}: {prob*100:.2f}%')

        st.warning('⚠️ This is an experimental AI prediction, not a medical diagnosis.')

In [ ]:
import os
from getpass import getpass
from pyngrok import ngrok

# Stop any existing sessions
ngrok.kill()
config_path = '/root/.config/ngrok/ngrok.yml'
if os.path.exists(config_path):
    os.remove(config_path)

# Enter your ngrok authtoken from https://dashboard.ngrok.com/
token = getpass('Enter your ngrok authtoken: ')
ngrok.set_auth_token(token.strip())
print('Token configured!')

In [ ]:
import time, requests

# Start Streamlit
!pkill -f streamlit 2>/dev/null || true
!streamlit run /content/app.py --server.address 127.0.0.1 --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &

# Wait for server to be ready
time.sleep(6)
resp = requests.get('http://127.0.0.1:8501/_stcore/health', timeout=10)
print('Streamlit status:', resp.text)

# Create ngrok tunnel
public_url = ngrok.connect(addr='http://127.0.0.1:8501', proto='http')
print('\n🧠 NeuroScan AI App:', public_url)